<p>Tests if the SPEI calculation implemented with scipy gives the same results as the SPEI Python package</p>

In [8]:
import pandas as pd
import numpy as np
import scipy.stats as st
import spei

In [3]:
test_csv =  "/home/politti/data/Danube_5min/spei_nuts_2/water_balance_danube_nuts2_cru.csv"
spatial_unit_col = "NUTS_ID"
test_nut = "AT11"

In [18]:
df = pd.read_csv(test_csv)
df = df[df[spatial_unit_col] == test_nut]
df = df[(df['year']>=1951) & (df['year']<=2010)]
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')
df = df[['water_balance']].copy(deep=True)
df.head()

,water_balance
date,
1951-01-16,1898.1001
1951-02-15,1227.2000
1951-03-16,3008.4000
1951-04-16,-2708.2998
1951-05-16,2246.9000


In [19]:
def calculate_spei(wb_series, scale=1):
    """
    Calculates SPEI by fitting the distribution to the entirety of the provided data.

    Parameters:
    wb_series (pd.Series): Monthly water balance (PR - PET) with a DatetimeIndex.
    scale (int): The accumulation timescale (e.g., 1 for 1-month, 3 for 3-month).

    Returns:
    pd.Series: Calculated SPEI values.
    """

    # 1. Handle the accumulation scale
    if scale > 1:
        data = wb_series.rolling(scale).sum().dropna()
    else:
        data = wb_series.dropna()

    # Empty series to store the results
    spei_result = pd.Series(index=data.index, dtype=float)

    # 2. Loop through each of the 12 months to handle seasonality
    for month in range(1, 13):

        # Extract data strictly for this calendar month
        month_data = data[data.index.month == month]

        # Skip if there isn't enough data to fit a distribution
        if len(month_data) < 2:
            continue

        # 3. Fit the Fisk (Log-Logistic) distribution
        params = st.fisk.fit(month_data)

        # 4. Calculate the Cumulative Distribution Function (CDF)
        cdf = st.fisk.cdf(month_data, *params)

        # 5. Transform the CDF to standard normal deviate (SPEI)
        # Clipping prevents 0 or 1 probabilities from becoming -inf or inf
        cdf = np.clip(cdf, 1e-6, 1 - 1e-6)
        spei_result.loc[month_data.index] = st.norm.ppf(cdf)

    return spei_result.sort_index()


def calc_spei_pck(wb_series, scale=1):
    if scale > 1:
        # Calculate the rolling sum for multi-month SPEI
        wb_prepared = wb_series.rolling(window=scale, min_periods=scale).sum().dropna()
    else:
        # For SPEI-1 (1-month), use the data as-is
        wb_prepared = wb_series.dropna()

    # 4. Calculate the SPEI!
    # The spei.spei() function automatically applies the Log-Logistic (Fisk) distribution
    # to each calendar month separately and returns a pandas Series.
    spei_series = spei.spei(wb_prepared)

    # 5. Merge the results back into your original dataframe
    return spei_series

In [21]:
wb_series = df['water_balance'].copy(deep=True)
spei_calc = calc_spei_pck(wb_series, scale = 12)

wb_series = df['water_balance'].copy(deep=True)
spei_calc_pck = calc_spei_pck(wb_series, scale = 12)

df['spei_calc'] = spei_calc
df['spei_calc_pck'] = spei_calc_pck

#df.dropna(inplace=True)

df[['spei_calc', 'spei_calc_pck']].head(24)




,spei_calc,spei_calc_pck
date,,
1951-03-16,NaN,NaN
1951-04-16,NaN,NaN
1951-05-16,NaN,NaN
1951-06-16,NaN,NaN
1951-07-16,NaN,NaN
1951-08-16,NaN,NaN
1951-09-16,NaN,NaN
1951-10-16,NaN,NaN
1951-11-16,NaN,NaN
